# AI Vocals Studio — real two-clip neural test
This notebook implements the production order: **separate → clone/convert isolated vocal → remix over the original separated beat → source-matched MP3**.

Default test direction: **The Uppercut = authorized target/reference voice** and **Only God Knows Why = source performance + original instrumental**. It never commits audio, stems, checkpoints, or outputs to GitHub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT = Path('/content/drive/MyDrive')
def find_one(name):
    hits=list(ROOT.rglob(name))
    if not hits: raise FileNotFoundError(f'{name} not found in MyDrive')
    print(name, '->', hits[0])
    return hits[0]
REFERENCE_SONG=find_one('The Uppercut.mp3')
SOURCE_SONG=find_one('Kid Rock - Only God Knows Why [Official Music Video].mp3')
OUT_DRIVE=ROOT/'AI_Vocals_Studio'/'two_clip_test'
OUT_DRIVE.mkdir(parents=True,exist_ok=True)


In [ ]:
import os, sys, subprocess, textwrap, shutil
subprocess.run(['apt-get','-qq','update'],check=True)
subprocess.run(['apt-get','-qq','install','-y','ffmpeg','git'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','demucs'],check=True)
if not Path('/content/ai-vocals-studio').exists(): subprocess.run(['git','clone','-q','https://github.com/coden607/ai-vocals-studio.git','/content/ai-vocals-studio'],check=True)
else: subprocess.run(['git','-C','/content/ai-vocals-studio','pull','-q','--ff-only'],check=True)
if not Path('/content/seed-vc').exists(): subprocess.run(['git','clone','-q','https://github.com/Plachtaa/seed-vc.git','/content/seed-vc'],check=True)
# Install Seed-VC's upstream dependencies while retaining Colab's working Torch stack.
req=[]
for line in Path('/content/seed-vc/requirements.txt').read_text().splitlines():
    s=line.strip()
    if not s or s.startswith('#') or s.startswith(('torch','torchvision','torchaudio')): continue
    req.append(s)
Path('/content/seed-vc/requirements-colab.txt').write_text('\n'.join(req)+'\n')
subprocess.run([sys.executable,'-m','pip','install','-q','-r','/content/seed-vc/requirements-colab.txt'],check=True)
os.environ['SEED_VC_DIR']='/content/seed-vc'
os.environ['SEED_VC_PYTHON']=sys.executable
sys.path.insert(0,'/content/ai-vocals-studio')
print('Runtime:', 'GPU' if __import__('torch').cuda.is_available() else 'CPU')


In [ ]:
# 1) Separate the target/reference song first. We use only its isolated vocal as the voice reference.
WORK=Path('/content/ai_vocals_test'); shutil.rmtree(WORK,ignore_errors=True); WORK.mkdir(parents=True)
ref_sep=WORK/'reference_sep'
subprocess.run([sys.executable,'-m','demucs','--two-stems','vocals','-n','htdemucs','-o',str(ref_sep),str(REFERENCE_SONG)],check=True)
ref_vocals=next(ref_sep.rglob('vocals.wav'))
print('Isolated target voice:',ref_vocals)


In [ ]:
# Pick the strongest 20-second vocal section automatically, avoiding quiet intro/outro material.
import numpy as np, librosa, soundfile as sf
y,sr=librosa.load(str(ref_vocals),sr=44100,mono=True)
win=20*sr; hop=5*sr
if len(y)<=win: start=0
else:
    starts=range(0,max(1,len(y)-win+1),hop)
    start=max(starts,key=lambda s: float(np.sqrt(np.mean(y[s:s+win]**2))))
ref_clip=WORK/'authorized_reference_voice.wav'
sf.write(ref_clip,y[start:start+win],sr)
print('Reference clip:',ref_clip,'start_s=',round(start/sr,1))


In [ ]:
# 2) Production song replacement. change_song itself separates the source song BEFORE conversion.
# Seed-VC receives only the isolated source vocal and the isolated authorized reference voice.
from song_converter import change_song
profile={'name':'UppercutVoice','reference':str(ref_clip),'reference_audio':str(ref_clip),'voice_dir':str(WORK/'voice')}
out,steps=change_song(str(SOURCE_SONG),profile,WORK/'render',separation='demucs',require_neural=True,source_matched_export=True)
print('PIPELINE STEPS:',steps)
final=Path(out)
assert final.exists() and final.stat().st_size>10000, final
dest=OUT_DRIVE/final.name
shutil.copy2(final,dest)
master=Path(steps.get('master',''))
if master.exists(): shutil.copy2(master,OUT_DRIVE/master.name)
print('DONE:',dest)


In [ ]:
# 3) Listen immediately in Colab; the same MP3 is saved in MyDrive/AI_Vocals_Studio/two_clip_test/.
from IPython.display import Audio, display
display(Audio(str(dest)))
print(dest)
